## Preprocesamiento y Modelado

Una vez inspeccionado el dataset en `customer_churn_eda.ipynb` definimos una una estrategia de preprocesamiento iterativo (de menos a más) para el encontrar

In [ ]:
import pandas as pd
import numpy as np
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.experimental import enable_iterative_imputer  # Necesario para IterativeImputer
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.preprocessing import PowerTransformer, RobustScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import cohen_kappa_score, f1_score, make_scorer
from sklearn.model_selection import cross_val_score, StratifiedKFold

# Rutas de los archivos de datos
TRAIN_PATH = "/kaggle/input/retencion-de-clientes-de-una-entidad-financiera/train.csv"
TEST_PATH = "/kaggle/input/retencion-de-clientes-de-una-entidad-financiera/test.csv"
SUBMISSION_PATH = "/kaggle/working/submission.csv"


# Cargamos de nueovo los datos para el pipeline
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
# Definir variables objetivo
TARGET = 'Exited'
# Variables numéricas: Incluyo las continuas y las binarias numéricas (HasCrCard, IsActiveMember)
NUM_FEATURES = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'EstimatedSalary', 'HasCrCard', 'IsActiveMember']
# Variables categóricas: Las de texto con pocas categorías
CAT_FEATURES = ['Geography', 'Gender']
# Variables a eliminar inicialmente (IDs y apellido)
DROP_FEATURES = ['CustomerId', 'Surname']
RANDOM_STATE = 100            # Semilla para reproducibilidad

# 1. Preparación de Datos
# Separar X_train e y_train por convención
# X_train = variables independientes
# y_train = variable dependiente u objetivo
X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET]

# Construcción de Pipelines de Preprocesamiento
# Dado que hay valores nulos en los datos,la imputación es obligatoria.

# Pipeline Numérico
num_pipeline = Pipeline(
    steps=[
    ('imputer', IterativeImputer(
        estimator=RandomForestRegressor(n_jobs=-1, random_state=RANDOM_STATE), # Modelo para estimar nulos
        max_iter=10, # Iteraciones para refinar
        random_state=RANDOM_STATE
    )),
    # Transformación: Yeo-Johnson intenta hacer la variable Gaussiana (normal)
    # Funciona mejor que logaritmo para datos con ceros y negativos.
    ('transformer', PowerTransformer(method='yeo-johnson')),
    # Escalado Robusto: Ignora outliers al escalar
    ('robust_scaler', RobustScaler())
])

# Pipeline Categórico
cat_pipeline = Pipeline(
    steps=[
    # Estrategia: Rellenar con el más frecuente (moda)
    ('imputer', SimpleImputer(strategy='most_frequent')),
    # Codificación: OneHot para convertir "France", "Germany" en números (0, 1)
    # handle_unknown='ignore' es CRÍTICO para que no falle si en test aparece algo raro
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# Preprocesador Maestro
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, NUM_FEATURES),
        ('cat', cat_pipeline, CAT_FEATURES)
    ],
    # 'remainder="drop"' elimina automáticamente CustomerId y Surname
    remainder='drop' 
)

# 3. Pipeline Completo (Preprocesamiento + Modelo)
# Empezamos con LinearDiscriminantAnalysis como línea base (simple y rápida) 
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=RANDOM_STATE, max_iter=1000,class_weight='balanced'))
])

# Validación Local  Cruzada (Fase 4 adelantada)
# Configuración: 5 splits (divisiones) .
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
# Definimos las métricas que queremos extraer
# f1, precision, recall, accuracy son strings estándar de sklearn.
# Kappa requiere make_scorer.
scoring_metrics = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'kappa': make_scorer(cohen_kappa_score)
}
# Ejecutar validación cruzada múltiple
cv_results = cross_validate(
    model_pipeline, 
    X_train, 
    y_train, 
    cv=cv_strategy,         # Estrategia de CV
    scoring=scoring_metrics,# Métricas a evaluar
    n_jobs=-1,              # Usar todos los núcleos disponibles
    return_train_score=True #  True para ver si hay overfitting
)

# Resultados medios
print("Resultados de Validación Cruzada:")
print(f"Mean F1-Score:  {cv_results['test_f1'].mean():.4f} (+/- Std {cv_results['test_f1'].std():.4f})")
print(f"Mean Accuracy:  {cv_results['test_accuracy'].mean():.4f} (+/- Std {cv_results['test_accuracy'].std():.4f})")
print(f"Mean Kappa:     {cv_results['test_kappa'].mean():.4f}")
print(f"Mean Precision: {cv_results['test_precision'].mean():.4f}")
print(f"Mean Recall:    {cv_results['test_recall'].mean():.4f}")



# 5. Generación de Submission para Kaggle (Fase 5)
# Re-entrenamos con TODOS los datos de train para la predicción final
model_pipeline.fit(X_train, y_train) 
test_predictions = model_pipeline.predict(test_df)

# Crear fichero de salida
submission = pd.DataFrame({
    'CustomerId': test_df['CustomerId'],
    'Exited': test_predictions
})
submission.to_csv(SUBMISSION_PATH, index=False)
print(f"Fichero '{SUBMISSION_PATH}' generado correctamente.")

ImportError: IterativeImputer is experimental and the API might change without any deprecation cycle. To use it, you need to explicitly import enable_iterative_imputer:
from sklearn.experimental import enable_iterative_imputer